# Agent-Enabled CLI for Cogitarelink
> Command-line interface designed specifically for agent interactions

In [ ]:
#| default_exp cli.agent_cli

In [ ]:
#| export
from __future__ import annotations
import sys
import json
import argparse
from typing import Dict, List, Any, Callable, Optional, Union
from pathlib import Path
from cogitarelink.cli.cli import Agent
from cogitarelink.core.debug import *
from cogitarelink.core.debug import get_logger

log = get_logger("cli.agent_cli")

In [ ]:
#| exporti
__all__ = ['AgentCLI', 'agent_cli_main']

# Agent-Optimized CLI

This notebook builds on the core CLI infrastructure to create a CLI that's optimized for agent interactions. 

Our key goals are:

1. **Automatic Tool Registration**: Include all available tools without manual configuration
2. **Agent-Friendly Output**: Structured JSON with agent guidance
3. **Category-Based Tool Organization**: Help agents discover relevant tools
4. **Earth616 Support**: First-class support for earth616_ontology as a test case
5. **Consistent Structure**: Predictable response format for reliable parsing

In [ ]:
# Fix the imports at the top of the file\nfrom cogitarelink.cli.cli import Agent, ToolRegistry\nfrom cogitarelink.core.debug import get_logger\nimport sys\nimport json\nimport argparse\nfrom typing import Dict, List, Any, Callable, Optional, Union\nfrom pathlib import Path\n\nlog = get_logger(\"agent_cli\")

In [ ]:
class AgentCLI(Agent):
    """Agent optimized for CLI interactions with other agents."""
    
    def __init__(self, name: str = "agent-cli"):
        super().__init__(name=name)
        self.context.remember("agent_mode", True)
        self._register_all_tools()
    
    def _register_all_tools(self):
        """Register all available tools from other agents."""
        # Register vocabulary tools
        try:
            from cogitarelink.cli.vocab_tools import VocabToolAgent
            vocab_agent = VocabToolAgent()
            
            # Register all vocab tools
            for name, tool in vocab_agent.tools.tools.items():
                if name not in self.tools.tools and name not in ["get_version", "list_tools", "get_agent_memory"]:
                    self.tools.tools[name] = tool
            
            self.context.remember("has_vocab_tools", True)
            log.info(f"Registered {len(vocab_agent.tools.tools) - 3} vocabulary tools")
        except Exception as e:
            self.context.remember("has_vocab_tools", False)
            self.context.remember("vocab_tools_error", str(e))
            log.error(f"Failed to register vocabulary tools: {e}")
        
        # Add earth616 specific tools
        self._register_earth616_tools()
    
    def _register_earth616_tools(self):
        """Register tools specifically for working with earth616 ontology."""
        
        @self.tools.register(name="register_earth616", 
                           description="Register earth616 ontology with the system")
        def register_earth616() -> Dict[str, Any]:
            """Register the earth616 ontology with the vocabulary registry."""
            
            CONTEXT_URL = "https://raw.githubusercontent.com/crcresearch/earth616_ontology/refs/heads/main/release/contexts/latest/context-base.jsonld"
            ONTOLOGY_URL = "https://raw.githubusercontent.com/crcresearch/earth616_ontology/refs/heads/main/release/ontology/latest/ontology.ttl"
            PREFIX = "earth616"
            URI_BASE = "https://ontology.crc.nd.edu/earth616/"
            
            # Step 1: Retrieve the context
            context_result = self.run_tool("retrieve_vocabulary_resource", uri=CONTEXT_URL)
            
            if not context_result.get("success", False):
                return {
                    "success": False,
                    "error": f"Failed to retrieve earth616 context: {context_result.get('error', 'Unknown error')}",
                    "agent_guidance": {
                        "description": "Failed to retrieve the earth616 context.",
                        "suggestions": [
                            "Check internet connection",
                            "Verify the earth616 context URL is accessible",
                            "Try again later"
                        ]
                    }
                }
            
            # Print debug info about the context result
            print(f"Context result keys: {context_result.keys()}")
            
            # Try different keys that might contain the context
            if "content" in context_result:
                context = context_result["content"]
            elif "data" in context_result:
                context = context_result["data"]
            else:
                # If we can't find the context, use context_url instead
                return {
                    "success": False,
                    "error": f"Could not extract context from result: {context_result}",
                    "agent_guidance": {
                        "description": "Failed to process the context data.",
                        "suggestions": [
                            "Try using context_url instead of inline_context"
                        ]
                    }
                }
            
            # Step 2: Register the vocabulary
            add_result = self.run_tool("add_temp_vocabulary", 
                                      prefix=PREFIX,
                                      uri=URI_BASE,
                                      context_url=CONTEXT_URL)
            
            if not add_result.get("success", False):
                return {
                    "success": False,
                    "error": f"Failed to register earth616 vocabulary: {add_result.get('error', 'Unknown error')}",
                    "agent_guidance": {
                        "description": "Failed to register the earth616 vocabulary.",
                        "suggestions": [
                            "Check if the vocabulary is already registered",
                            "Try again with a different prefix if there's a conflict"
                        ]
                    }
                }
            
            # Step 3: Store information about the SHACL shapes
            self.context.remember("earth616_shapes_url", 
                                "https://raw.githubusercontent.com/crcresearch/earth616_ontology/main/release/shapes/shacl/latest/shapes.ttl")
            
            # Step 4: Return success response with guidance
            return {
                "success": True,
                "message": f"Successfully registered earth616 ontology with prefix '{PREFIX}'",
                "context_size": len(str(context)),
                "agent_guidance": {
                    "description": "The earth616 ontology is now registered in the system.",
                    "next_steps": [
                        "Get vocabulary information: cogitarelink --run-tool get_vocabulary_info --args '{\"prefix\":\"earth616\"}'",
                        "Compose a context: cogitarelink --run-tool compose_context --args '{\"prefixes\":[\"earth616\"]}'",
                        "Generate an example: cogitarelink --run-tool generate_earth616_example"
                    ]
                }
            }
        
        @self.tools.register(name="generate_earth616_example",
                           description="Generate an example entity using the earth616 ontology")
        def generate_earth616_example(entity_type: str = "Event") -> Dict[str, Any]:
            """Generate an example entity using the earth616 ontology."""
            
            # Step 1: Check if earth616 is registered, if not register it
            vocab_info = self.run_tool("get_vocabulary_info", prefix="earth616")
            
            if not vocab_info.get("success", False):
                print("Earth616 vocabulary not found, auto-registering...")
                register_result = self.run_tool("register_earth616")
                
                if not register_result.get("success", False):
                    return {
                        "success": False,
                        "error": f"Failed to auto-register earth616: {register_result.get('error', 'Unknown error')}",
                        "agent_guidance": {
                            "description": "Failed to automatically register the earth616 vocabulary.",
                            "suggestions": [
                                "Try explicitly running: cogitarelink --run-tool register_earth616"
                            ]
                        }
                    }
            
            # Step 2: Create a simple context directly
            context_obj = {
                "@context": {
                    "@vocab": "https://ontology.crc.nd.edu/earth616/"
                }
            }
            
            # Print debug info
            print(f"Using simplified context: {context_obj}")
            
            # Step 3: Create an example entity
            example_data = {
                "@context": context_obj,
                "@type": entity_type,
                "@id": f"urn:example:{entity_type.lower()}:1234",
                "name": f"Example {entity_type}",
                "description": f"A sample {entity_type} using the Earth616 ontology context"
            }
            
            # Add type-specific properties
            if entity_type == "Event":
                example_data.update({
                    "startTime": "2023-01-01T00:00:00Z",
                    "endTime": "2023-01-02T00:00:00Z",
                    "location": {
                        "@type": "Place",
                        "name": "Example Location",
                        "address": {
                            "@type": "PostalAddress",
                            "addressLocality": "South Bend",
                            "addressRegion": "IN",
                            "addressCountry": "US"
                        }
                    },
                    "participant": [
                        {
                            "@type": "Organization",
                            "@id": "urn:example:org:1",
                            "name": "Example Organization"
                        }
                    ]
                })
            elif entity_type == "Place":
                example_data.update({
                    "geo": {
                        "@type": "GeoCoordinates",
                        "latitude": 41.6764,
                        "longitude": -86.2520
                    },
                    "address": {
                        "@type": "PostalAddress",
                        "addressLocality": "South Bend",
                        "addressRegion": "IN",
                        "addressCountry": "US"
                    }
                })
            elif entity_type == "Person" or entity_type == "Organization":
                example_data.update({
                    "url": f"https://example.org/{entity_type.lower()}/1234",
                    "email": f"info@example.org"
                })
            
            # Return the example with guidance
            return {
                "success": True,
                "example": example_data,
                "agent_guidance": {
                    "description": f"Generated an example {entity_type} using earth616 ontology.",
                    "next_steps": [
                        "Convert to other formats: cogitarelink --run-tool convert_format --args '{\"content\": \"<example>\", \"from_format\": \"json-ld\", \"to_format\": \"turtle\"}'",
                        "Generate examples of other types: cogitarelink --run-tool generate_earth616_example --args '{\"entity_type\": \"Organization\"}'"
                    ],
                    "available_types": ["Event", "Place", "Person", "Organization", "Product"]
                }
            }
            
    def categorize_tools(self) -> Dict[str, List[Dict[str, Any]]]:
        """Categorize all tools by their purpose."""
        # Get all tools
        all_tools = self.run_tool("list_tools")
        
        # Define categories
        categories = {
            "core": [],
            "registry": [],
            "context": [],
            "retrieval": [],
            "validation": [],
            "earth616": []
        }
        
        # Categorize each tool
        for tool in all_tools:
            name = tool.get("name", "")
            desc = tool.get("description", "").lower()
            
            # Core tools
            if name in ["get_version", "list_tools", "get_agent_memory"]:
                categories["core"].append(tool)
            # Earth616 tools
            elif "earth616" in name or "earth616" in desc:
                categories["earth616"].append(tool)
            # Registry tools
            elif any(x in name for x in ["registry", "vocabulary", "vocab"]) or \
                 any(x in desc for x in ["registry", "vocabulary", "vocab"]):
                categories["registry"].append(tool)
            # Context tools
            elif any(x in name for x in ["context", "compose"]) or \
                 any(x in desc for x in ["context", "compose"]):
                categories["context"].append(tool)
            # Retrieval tools
            elif any(x in name for x in ["retrieve", "fetch", "search", "convert"]) or \
                 any(x in desc for x in ["retrieve", "fetch", "search", "convert"]):
                categories["retrieval"].append(tool)
            # Validation tools
            elif any(x in name for x in ["validate", "verify", "check", "shape"]) or \
                 any(x in desc for x in ["validate", "verify", "check", "shape"]):
                categories["validation"].append(tool)
            # Default to core
            else:
                categories["core"].append(tool)
        
        return categories
    
    def run_tool_with_guidance(self, name: str, **kwargs) -> Dict[str, Any]:
        """Run a tool and add agent guidance to the result."""
        try:
            # Run the tool
            result = self.run_tool(name, **kwargs)
            
            # If already has agent guidance, return as is
            if isinstance(result, dict) and "agent_guidance" in result:
                return result
            
            # Create guidance based on the tool and result
            guidance = {
                "description": f"Successfully ran tool: {name}"
            }
            
            # Add next steps based on the tool
            if name == "retrieve_vocabulary_resource":
                guidance["next_steps"] = [
                    "Register this as a vocabulary: cogitarelink --run-tool add_temp_vocabulary"
                ]
            elif name == "add_temp_vocabulary":
                guidance["next_steps"] = [
                    "Get vocabulary info: cogitarelink --run-tool get_vocabulary_info",
                    "Compose a context: cogitarelink --run-tool compose_context"
                ]
            elif name == "compose_context":
                guidance["next_steps"] = [
                    "Use this context in your JSON-LD documents"
                ]
            elif name == "convert_format":
                guidance["next_steps"] = [
                    "You can convert between various formats including: json-ld, turtle, n3, xml"
                ]
            
            # If result is a dict, add the guidance directly
            if isinstance(result, dict):
                result["agent_guidance"] = guidance
                return result
            else:
                # Wrap the result in a dict with guidance
                return {
                    "success": True,
                    "result": result,
                    "agent_guidance": guidance
                }
        except Exception as e:
            # Create an error response with guidance
            return {
                "success": False,
                "error": f"Error running tool {name}: {str(e)}",
                "agent_guidance": {
                    "description": f"There was an error running the tool '{name}'.",
                    "suggestions": [
                        f"Check if '{name}' exists: cogitarelink --list-tools",
                        "Check that all required arguments are provided",
                        "If working with vocabularies, make sure they are registered first"
                    ]
                }
            }

def agent_cli_main():
    """Main entry point for the agent-optimized CLI."""
    # Skip argument parsing when run inside a notebook
    in_notebook = 'ipykernel' in sys.modules
    
    if in_notebook:
        # Default behavior for notebook - create an agent and return it
        return AgentCLI()
    
    # Create our agent-optimized CLI
    agent = AgentCLI()
    
    # Parse arguments
    parser = argparse.ArgumentParser(description="CogitareLink: Semantic Memory Tools for Agents")
    parser.add_argument('--version', action='store_true', help='Show version and exit')
    parser.add_argument('--list-tools', action='store_true', help='List available tools')
    parser.add_argument('--category', help='Filter tools by category (registry, context, retrieval, validation, earth616)')
    parser.add_argument('--run-tool', metavar='TOOL', help='Run a specific tool')
    parser.add_argument('--args', metavar='JSON', help='JSON arguments for tool')
    parser.add_argument('--args-file', metavar='FILE', help='File containing JSON arguments for tool')
    
    args = parser.parse_args()
    
    if args.version:
        version = agent.run_tool('get_version')
        result = {
            "version": version,
            "agent_guidance": {
                "description": "CogitareLink is a tool for working with Linked Open Data as semantic memory.",
                "next_steps": [
                    "List available tools: cogitarelink --list-tools",
                    "Register earth616 ontology: cogitarelink --run-tool register_earth616"
                ]
            }
        }
        print(json.dumps(result, indent=2))
        return 0
        
    if args.list_tools:
        # Get tools by category
        categories = agent.categorize_tools()
        
        # Filter by category if requested
        if args.category and args.category in categories:
            tools_dict = {args.category: categories[args.category]}
        else:
            tools_dict = categories
        
        # Format for agent consumption
        result = {
            "tools_by_category": {}
        }
        
        for category, tools in tools_dict.items():
            result["tools_by_category"][category] = [
                {
                    "name": tool["name"],
                    "description": tool["description"],
                    "arguments": tool.get("signature", {})
                } for tool in tools
            ]
        
        # Add agent guidance
        result["agent_guidance"] = {
            "description": "These are the available tools in CogitareLink.",
            "usage": "To use a tool: cogitarelink --run-tool TOOL_NAME --args '{\"arg1\": \"value1\"}'",
            "earth616_workflow": [
                "1. Register earth616: cogitarelink --run-tool register_earth616",
                "2. Generate example data: cogitarelink --run-tool generate_earth616_example",
                "3. Convert to other formats: cogitarelink --run-tool convert_format"
            ]
        }
        
        print(json.dumps(result, indent=2))
        return 0
        
    if args.run_tool:
        tool_name = args.run_tool
        tool_args = {}
        
        # Get tool args from --args or --args-file
        if args.args:
            try:
                tool_args = json.loads(args.args)
            except json.JSONDecodeError as e:
                error_result = {
                    "success": False,
                    "error": f"Error parsing JSON arguments: {str(e)}",
                    "agent_guidance": {
                        "description": "There was an error parsing the JSON arguments.",
                        "suggestions": [
                            "Check JSON syntax and try again",
                            "For complex or large arguments, use --args-file instead"
                        ]
                    }
                }
                print(json.dumps(error_result, indent=2))
                return 1
        elif args.args_file:
            try:
                with open(args.args_file, 'r') as f:
                    tool_args = json.loads(f.read())
            except FileNotFoundError:
                error_result = {
                    "success": False,
                    "error": f"Arguments file not found: {args.args_file}",
                    "agent_guidance": {
                        "description": "The specified arguments file was not found.",
                        "suggestions": [
                            "Check the file path and try again"
                        ]
                    }
                }
                print(json.dumps(error_result, indent=2))
                return 1
            except json.JSONDecodeError as e:
                error_result = {
                    "success": False,
                    "error": f"Error parsing JSON in arguments file: {str(e)}",
                    "agent_guidance": {
                        "description": "There was an error parsing the JSON in the arguments file.",
                        "suggestions": [
                            "Check JSON syntax in the file and try again"
                        ]
                    }
                }
                print(json.dumps(error_result, indent=2))
                return 1
        
        # Run the tool with guidance
        result = agent.run_tool_with_guidance(tool_name, **tool_args)
        print(json.dumps(result, indent=2, default=str))
        return 0 if result.get("success", False) else 1
    
    # Default behavior - show help with agent guidance
    parser.print_help()
    
    # Add agent guidance
    help_guidance = {
        "agent_guidance": {
            "description": "CogitareLink is a tool for working with Linked Open Data as semantic memory.",
            "available_commands": [
                "--list-tools: List all available tools",
                "--category CATEGORY: Filter tools by category (registry, context, retrieval, validation, earth616)",
                "--run-tool TOOL: Run a specific tool",
                "--args '{\"arg1\": \"value1\"}': Provide JSON arguments for the tool",
                "--args-file file.json: Provide arguments from a JSON file"
            ],
            "getting_started": [
                "1. List available tools: cogitarelink --list-tools",
                "2. Register earth616 ontology: cogitarelink --run-tool register_earth616",
                "3. Generate an example: cogitarelink --run-tool generate_earth616_example"
            ]
        }
    }
    
    print("\n" + json.dumps(help_guidance, indent=2))
    return 0

In [ ]:
#| export
class AgentCLI(Agent):
    """Agent optimized for CLI interactions with other agents."""
    
    def __init__(self, name: str = "agent-cli"):
        super().__init__(name=name)
        self.context.remember("agent_mode", True)
        self._register_all_tools()
    
    def _register_all_tools(self):
        """Register all available tools from other agents."""
        # Register vocabulary tools
        try:
            from cogitarelink.cli.vocab_tools import VocabToolAgent
            vocab_agent = VocabToolAgent()
            
            # Register all vocab tools
            for name, tool in vocab_agent.tools.tools.items():
                if name not in self.tools.tools and name not in ["get_version", "list_tools", "get_agent_memory"]:
                    self.tools.tools[name] = tool
            
            self.context.remember("has_vocab_tools", True)
            log.info(f"Registered {len(vocab_agent.tools.tools) - 3} vocabulary tools")
        except Exception as e:
            self.context.remember("has_vocab_tools", False)
            self.context.remember("vocab_tools_error", str(e))
            log.error(f"Failed to register vocabulary tools: {e}")
        
        # Add earth616 specific tools
        self._register_earth616_tools()
    
    def _register_earth616_tools(self):
        """Register tools specifically for working with earth616 ontology."""
        
        @self.tools.register(name="register_earth616", 
                           description="Register earth616 ontology with the system")
        def register_earth616() -> Dict[str, Any]:
            """Register the earth616 ontology with the vocabulary registry."""
            
            CONTEXT_URL = "https://raw.githubusercontent.com/crcresearch/earth616_ontology/refs/heads/main/release/contexts/latest/context-base.jsonld"
            ONTOLOGY_URL = "https://raw.githubusercontent.com/crcresearch/earth616_ontology/refs/heads/main/release/ontology/latest/ontology.ttl"
            PREFIX = "earth616"
            URI_BASE = "https://ontology.crc.nd.edu/earth616/"
            
            # Step 1: Retrieve the context
            context_result = self.run_tool("retrieve_vocabulary_resource", uri=CONTEXT_URL)
            
            if not context_result.get("success", False):
                return {
                    "success": False,
                    "error": f"Failed to retrieve earth616 context: {context_result.get('error', 'Unknown error')}",
                    "agent_guidance": {
                        "description": "Failed to retrieve the earth616 context.",
                        "suggestions": [
                            "Check internet connection",
                            "Verify the earth616 context URL is accessible",
                            "Try again later"
                        ]
                    }
                }
            
            # Print debug info about the context result
            print(f"Context result keys: {context_result.keys()}")
            
            # Try different keys that might contain the context
            if "content" in context_result:
                context = context_result["content"]
            elif "data" in context_result:
                context = context_result["data"]
            else:
                # If we can't find the context, use context_url instead
                return {
                    "success": False,
                    "error": f"Could not extract context from result: {context_result}",
                    "agent_guidance": {
                        "description": "Failed to process the context data.",
                        "suggestions": [
                            "Try using context_url instead of inline_context"
                        ]
                    }
                }
            
            # Step 2: Register the vocabulary
            add_result = self.run_tool("add_temp_vocabulary", 
                                      prefix=PREFIX,
                                      uri=URI_BASE,
                                      context_url=CONTEXT_URL)
            
            if not add_result.get("success", False):
                return {
                    "success": False,
                    "error": f"Failed to register earth616 vocabulary: {add_result.get('error', 'Unknown error')}",
                    "agent_guidance": {
                        "description": "Failed to register the earth616 vocabulary.",
                        "suggestions": [
                            "Check if the vocabulary is already registered",
                            "Try again with a different prefix if there's a conflict"
                        ]
                    }
                }
            
            # Step 3: Store information about the SHACL shapes
            self.context.remember("earth616_shapes_url", 
                                "https://raw.githubusercontent.com/crcresearch/earth616_ontology/main/release/shapes/shacl/latest/shapes.ttl")
            
            # Step 4: Return success response with guidance
            return {
                "success": True,
                "message": f"Successfully registered earth616 ontology with prefix '{PREFIX}'",
                "context_size": len(str(context)),
                "agent_guidance": {
                    "description": "The earth616 ontology is now registered in the system.",
                    "next_steps": [
                        "Get vocabulary information: cogitarelink --run-tool get_vocabulary_info --args '{\"prefix\":\"earth616\"}'",
                        "Compose a context: cogitarelink --run-tool compose_context --args '{\"prefixes\":[\"earth616\"]}'",
                        "Generate an example: cogitarelink --run-tool generate_earth616_example"
                    ]
                }
            }
        
        @self.tools.register(name="generate_earth616_example",
                           description="Generate an example entity using the earth616 ontology")
        def generate_earth616_example(entity_type: str = "Event") -> Dict[str, Any]:
            """Generate an example entity using the earth616 ontology."""
            
            # Step 1: Check if earth616 is registered, if not register it
            vocab_info = self.run_tool("get_vocabulary_info", prefix="earth616")
            
            if not vocab_info.get("success", False):
                print("Earth616 vocabulary not found, auto-registering...")
                register_result = self.run_tool("register_earth616")
                
                if not register_result.get("success", False):
                    return {
                        "success": False,
                        "error": f"Failed to auto-register earth616: {register_result.get('error', 'Unknown error')}",
                        "agent_guidance": {
                            "description": "Failed to automatically register the earth616 vocabulary.",
                            "suggestions": [
                                "Try explicitly running: cogitarelink --run-tool register_earth616"
                            ]
                        }
                    }
            
            # Step 2: Create a simple context directly
            context_obj = {
                "@context": {
                    "@vocab": "https://ontology.crc.nd.edu/earth616/"
                }
            }
            
            # Print debug info
            print(f"Using simplified context: {context_obj}")
            
            # Step 3: Create an example entity
            example_data = {
                "@context": context_obj,
                "@type": entity_type,
                "@id": f"urn:example:{entity_type.lower()}:1234",
                "name": f"Example {entity_type}",
                "description": f"A sample {entity_type} using the Earth616 ontology context"
            }
            
            # Add type-specific properties
            if entity_type == "Event":
                example_data.update({
                    "startTime": "2023-01-01T00:00:00Z",
                    "endTime": "2023-01-02T00:00:00Z",
                    "location": {
                        "@type": "Place",
                        "name": "Example Location",
                        "address": {
                            "@type": "PostalAddress",
                            "addressLocality": "South Bend",
                            "addressRegion": "IN",
                            "addressCountry": "US"
                        }
                    },
                    "participant": [
                        {
                            "@type": "Organization",
                            "@id": "urn:example:org:1",
                            "name": "Example Organization"
                        }
                    ]
                })
            elif entity_type == "Place":
                example_data.update({
                    "geo": {
                        "@type": "GeoCoordinates",
                        "latitude": 41.6764,
                        "longitude": -86.2520
                    },
                    "address": {
                        "@type": "PostalAddress",
                        "addressLocality": "South Bend",
                        "addressRegion": "IN",
                        "addressCountry": "US"
                    }
                })
            elif entity_type == "Person" or entity_type == "Organization":
                example_data.update({
                    "url": f"https://example.org/{entity_type.lower()}/1234",
                    "email": f"info@example.org"
                })
            
            # Return the example with guidance
            return {
                "success": True,
                "example": example_data,
                "agent_guidance": {
                    "description": f"Generated an example {entity_type} using earth616 ontology.",
                    "next_steps": [
                        "Convert to other formats: cogitarelink --run-tool convert_format --args '{\"content\": \"<example>\", \"from_format\": \"json-ld\", \"to_format\": \"turtle\"}'",
                        "Generate examples of other types: cogitarelink --run-tool generate_earth616_example --args '{\"entity_type\": \"Organization\"}'"
                    ],
                    "available_types": ["Event", "Place", "Person", "Organization", "Product"]
                }
            }
            
    def categorize_tools(self) -> Dict[str, List[Dict[str, Any]]]:
        """Categorize all tools by their purpose."""
        # Get all tools
        all_tools = self.run_tool("list_tools")
        
        # Define categories
        categories = {
            "core": [],
            "registry": [],
            "context": [],
            "retrieval": [],
            "validation": [],
            "earth616": []
        }
        
        # Categorize each tool
        for tool in all_tools:
            name = tool.get("name", "")
            desc = tool.get("description", "").lower()
            
            # Core tools
            if name in ["get_version", "list_tools", "get_agent_memory"]:
                categories["core"].append(tool)
            # Earth616 tools
            elif "earth616" in name or "earth616" in desc:
                categories["earth616"].append(tool)
            # Registry tools
            elif any(x in name for x in ["registry", "vocabulary", "vocab"]) or \
                 any(x in desc for x in ["registry", "vocabulary", "vocab"]):
                categories["registry"].append(tool)
            # Context tools
            elif any(x in name for x in ["context", "compose"]) or \
                 any(x in desc for x in ["context", "compose"]):
                categories["context"].append(tool)
            # Retrieval tools
            elif any(x in name for x in ["retrieve", "fetch", "search", "convert"]) or \
                 any(x in desc for x in ["retrieve", "fetch", "search", "convert"]):
                categories["retrieval"].append(tool)
            # Validation tools
            elif any(x in name for x in ["validate", "verify", "check", "shape"]) or \
                 any(x in desc for x in ["validate", "verify", "check", "shape"]):
                categories["validation"].append(tool)
            # Default to core
            else:
                categories["core"].append(tool)
        
        return categories
    
    def run_tool_with_guidance(self, name: str, **kwargs) -> Dict[str, Any]:
        """Run a tool and add agent guidance to the result."""
        try:
            # Run the tool
            result = self.run_tool(name, **kwargs)
            
            # If already has agent guidance, return as is
            if isinstance(result, dict) and "agent_guidance" in result:
                return result
            
            # Create guidance based on the tool and result
            guidance = {
                "description": f"Successfully ran tool: {name}"
            }
            
            # Add next steps based on the tool
            if name == "retrieve_vocabulary_resource":
                guidance["next_steps"] = [
                    "Register this as a vocabulary: cogitarelink --run-tool add_temp_vocabulary"
                ]
            elif name == "add_temp_vocabulary":
                guidance["next_steps"] = [
                    "Get vocabulary info: cogitarelink --run-tool get_vocabulary_info",
                    "Compose a context: cogitarelink --run-tool compose_context"
                ]
            elif name == "compose_context":
                guidance["next_steps"] = [
                    "Use this context in your JSON-LD documents"
                ]
            elif name == "convert_format":
                guidance["next_steps"] = [
                    "You can convert between various formats including: json-ld, turtle, n3, xml"
                ]
            
            # If result is a dict, add the guidance directly
            if isinstance(result, dict):
                result["agent_guidance"] = guidance
                return result
            else:
                # Wrap the result in a dict with guidance
                return {
                    "success": True,
                    "result": result,
                    "agent_guidance": guidance
                }
        except Exception as e:
            # Create an error response with guidance
            return {
                "success": False,
                "error": f"Error running tool {name}: {str(e)}",
                "agent_guidance": {
                    "description": f"There was an error running the tool '{name}'.",
                    "suggestions": [
                        f"Check if '{name}' exists: cogitarelink --list-tools",
                        "Check that all required arguments are provided",
                        "If working with vocabularies, make sure they are registered first"
                    ]
                }
            }

## Command Line Interface

Now, let's create the main CLI function that uses our agent:

In [ ]:
#| export
def agent_cli_main():
    """Main entry point for the agent-optimized CLI."""
    # Skip argument parsing when run inside a notebook
    in_notebook = 'ipykernel' in sys.modules
    
    if in_notebook:
        # Default behavior for notebook - create an agent and return it
        return AgentCLI()
    
    # Create our agent-optimized CLI
    agent = AgentCLI()
    
    # Parse arguments
    parser = argparse.ArgumentParser(description="CogitareLink: Semantic Memory Tools for Agents")
    parser.add_argument('--version', action='store_true', help='Show version and exit')
    parser.add_argument('--list-tools', action='store_true', help='List available tools')
    parser.add_argument('--category', help='Filter tools by category (registry, context, retrieval, validation, earth616)')
    parser.add_argument('--run-tool', metavar='TOOL', help='Run a specific tool')
    parser.add_argument('--args', metavar='JSON', help='JSON arguments for tool')
    parser.add_argument('--args-file', metavar='FILE', help='File containing JSON arguments for tool')
    
    args = parser.parse_args()
    
    if args.version:
        version = agent.run_tool('get_version')
        result = {
            "version": version,
            "agent_guidance": {
                "description": "CogitareLink is a tool for working with Linked Open Data as semantic memory.",
                "next_steps": [
                    "List available tools: cogitarelink --list-tools",
                    "Register earth616 ontology: cogitarelink --run-tool register_earth616"
                ]
            }
        }
        print(json.dumps(result, indent=2))
        return 0
        
    if args.list_tools:
        # Get tools by category
        categories = agent.categorize_tools()
        
        # Filter by category if requested
        if args.category and args.category in categories:
            tools_dict = {args.category: categories[args.category]}
        else:
            tools_dict = categories
        
        # Format for agent consumption
        result = {
            "tools_by_category": {}
        }
        
        for category, tools in tools_dict.items():
            result["tools_by_category"][category] = [
                {
                    "name": tool["name"],
                    "description": tool["description"],
                    "arguments": tool.get("signature", {})
                } for tool in tools
            ]
        
        # Add agent guidance
        result["agent_guidance"] = {
            "description": "These are the available tools in CogitareLink.",
            "usage": "To use a tool: cogitarelink --run-tool TOOL_NAME --args '{\"arg1\": \"value1\"}'",
            "earth616_workflow": [
                "1. Register earth616: cogitarelink --run-tool register_earth616",
                "2. Generate example data: cogitarelink --run-tool generate_earth616_example",
                "3. Convert to other formats: cogitarelink --run-tool convert_format"
            ]
        }
        
        print(json.dumps(result, indent=2))
        return 0
        
    if args.run_tool:
        tool_name = args.run_tool
        tool_args = {}
        
        # Get tool args from --args or --args-file
        if args.args:
            try:
                tool_args = json.loads(args.args)
            except json.JSONDecodeError as e:
                error_result = {
                    "success": False,
                    "error": f"Error parsing JSON arguments: {str(e)}",
                    "agent_guidance": {
                        "description": "There was an error parsing the JSON arguments.",
                        "suggestions": [
                            "Check JSON syntax and try again",
                            "For complex or large arguments, use --args-file instead"
                        ]
                    }
                }
                print(json.dumps(error_result, indent=2))
                return 1
        elif args.args_file:
            try:
                with open(args.args_file, 'r') as f:
                    tool_args = json.loads(f.read())
            except FileNotFoundError:
                error_result = {
                    "success": False,
                    "error": f"Arguments file not found: {args.args_file}",
                    "agent_guidance": {
                        "description": "The specified arguments file was not found.",
                        "suggestions": [
                            "Check the file path and try again"
                        ]
                    }
                }
                print(json.dumps(error_result, indent=2))
                return 1
            except json.JSONDecodeError as e:
                error_result = {
                    "success": False,
                    "error": f"Error parsing JSON in arguments file: {str(e)}",
                    "agent_guidance": {
                        "description": "There was an error parsing the JSON in the arguments file.",
                        "suggestions": [
                            "Check JSON syntax in the file and try again"
                        ]
                    }
                }
                print(json.dumps(error_result, indent=2))
                return 1
        
        # Run the tool with guidance
        result = agent.run_tool_with_guidance(tool_name, **tool_args)
        print(json.dumps(result, indent=2, default=str))
        return 0 if result.get("success", False) else 1
    
    # Default behavior - show help with agent guidance
    parser.print_help()
    
    # Add agent guidance
    help_guidance = {
        "agent_guidance": {
            "description": "CogitareLink is a tool for working with Linked Open Data as semantic memory.",
            "available_commands": [
                "--list-tools: List all available tools",
                "--category CATEGORY: Filter tools by category (registry, context, retrieval, validation, earth616)",
                "--run-tool TOOL: Run a specific tool",
                "--args '{\"arg1\": \"value1\"}': Provide JSON arguments for the tool",
                "--args-file file.json: Provide arguments from a JSON file"
            ],
            "getting_started": [
                "1. List available tools: cogitarelink --list-tools",
                "2. Register earth616 ontology: cogitarelink --run-tool register_earth616",
                "3. Generate an example: cogitarelink --run-tool generate_earth616_example"
            ]
        }
    }
    
    print("\n" + json.dumps(help_guidance, indent=2))
    return 0



# Testing the Agent CLI

Let's create an instance of our agent CLI and test its basic functionality:

In [ ]:
agent_cli = AgentCLI()

# Test categorization of tools
categories = agent_cli.categorize_tools()
for category, tools in categories.items():
    print(f"{category}: {len(tools)} tools")

core: 9 tools
registry: 9 tools
context: 0 tools
retrieval: 2 tools
validation: 0 tools
earth616: 2 tools


In [ ]:
# Test earth616 tools
for tool in categories.get("earth616", []):
    print(f"{tool['name']}: {tool['description']}")

register_earth616: Register earth616 ontology with the system
generate_earth616_example: Generate an example entity using the earth616 ontology


# Next Steps

This implementation provides a minimal but functional agent-enabled CLI with the following features:

1. **Automatic Tool Registration**: Imports and registers all tools from VocabToolAgent
2. **Earth616-Specific Tools**: Special tools for easily working with earth616_ontology
3. **Agent Guidance**: Structured output with guidance on next steps
4. **Category Organization**: Tools organized by category for better discovery
5. **Support for Arguments File**: Ability to pass large JSON arguments via file

After testing and refinement, we should expand this with more validation tools and more sophisticated SHACL and inference capabilities.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()